# Grad-CAM: Visualizing Where the Model Looks

Use Gradient-weighted Class Activation Mapping (Grad-CAM) to highlight which spatial regions
of the input image are most important for the model's prediction.

In [ ]:
import torch
import torch.nn.functional as F
import timm
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets
from torch.utils.data import DataLoader
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

NUM_CLASSES = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(torch.load("fids30_classifier_30cls_b0.pth", map_location=device))
model = model.to(device)
model.eval()

config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**config, is_training=False)
test_ds = datasets.ImageFolder("PrepData/Test", transform=transform)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

class_names = test_ds.classes
print(f"Device: {device}")
print(f"Test samples: {len(test_ds)}")
print(f"Classes ({len(class_names)}): {class_names}")

## 1. Collect Predictions

In [ ]:
all_labels = []
all_preds = []
all_images = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1)
        all_labels.append(labels)
        all_preds.append(preds.cpu())
        all_images.append(images.cpu())

all_labels = torch.cat(all_labels)
all_preds = torch.cat(all_preds)
all_images = torch.cat(all_images)

correct_mask = all_preds == all_labels
n_correct = correct_mask.sum().item()
n_wrong = (~correct_mask).sum().item()
print(f"Correct: {n_correct}, Misclassified: {n_wrong} ({100*n_wrong/len(all_labels):.1f}%)")

## 2. Grad-CAM Setup

Target the last convolutional block of EfficientNet-B0 for the activation maps.

In [ ]:
# The last conv layer in EfficientNet-B0 (timm) is model.conv_head
target_layers = [model.conv_head]

cam_gradcam = GradCAM(model=model, target_layers=target_layers)
cam_gradcampp = GradCAMPlusPlus(model=model, target_layers=target_layers)

def denorm_image(img_tensor):
    """Convert normalized tensor back to displayable [0,1] RGB image."""
    img = img_tensor.numpy().transpose(1, 2, 0)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return np.float32(img)

print(f"Target layer: {target_layers[0].__class__.__name__}")
print("Grad-CAM and Grad-CAM++ ready.")

## 3. Grad-CAM: Misclassified Samples

For misclassified samples, show Grad-CAM for both the predicted and true class to compare what the model focused on.

In [ ]:
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def plot_gradcam(indices, title, show_true_class=False, max_show=8):
    """Plot Grad-CAM and Grad-CAM++ visualizations."""
    indices = indices[:max_show]
    n = len(indices)
    if n == 0:
        print(f"No samples to show for: {title}")
        return

    ncols = 5 if show_true_class else 3
    fig, axes = plt.subplots(n, ncols, figsize=(4 * ncols, 3.5 * n))
    fig.suptitle(title, fontsize=14, y=1.01)
    if n == 1:
        axes = axes.reshape(1, ncols)

    for row, idx in enumerate(indices):
        true_label = all_labels[idx].item()
        pred_label = all_preds[idx].item()
        img_tensor = all_images[idx].unsqueeze(0)
        img_display = denorm_image(all_images[idx])

        # Grad-CAM for predicted class
        targets_pred = [ClassifierOutputTarget(pred_label)]
        gc_pred = cam_gradcam(input_tensor=img_tensor, targets=targets_pred)[0]
        gcpp_pred = cam_gradcampp(input_tensor=img_tensor, targets=targets_pred)[0]

        overlay_gc = show_cam_on_image(img_display, gc_pred, use_rgb=True)
        overlay_gcpp = show_cam_on_image(img_display, gcpp_pred, use_rgb=True)

        # Original
        axes[row, 0].imshow(img_display)
        axes[row, 0].set_title(f"True: {class_names[true_label]}", fontsize=10)
        axes[row, 0].axis('off')

        # Grad-CAM (predicted)
        axes[row, 1].imshow(overlay_gc)
        axes[row, 1].set_title(f"Grad-CAM (pred: {class_names[pred_label]})", fontsize=10)
        axes[row, 1].axis('off')

        # Grad-CAM++ (predicted)
        axes[row, 2].imshow(overlay_gcpp)
        axes[row, 2].set_title(f"Grad-CAM++ (pred: {class_names[pred_label]})", fontsize=10)
        axes[row, 2].axis('off')

        if show_true_class:
            # Grad-CAM for true class
            targets_true = [ClassifierOutputTarget(true_label)]
            gc_true = cam_gradcam(input_tensor=img_tensor, targets=targets_true)[0]
            gcpp_true = cam_gradcampp(input_tensor=img_tensor, targets=targets_true)[0]

            axes[row, 3].imshow(show_cam_on_image(img_display, gc_true, use_rgb=True))
            axes[row, 3].set_title(f"Grad-CAM (true: {class_names[true_label]})", fontsize=10)
            axes[row, 3].axis('off')

            axes[row, 4].imshow(show_cam_on_image(img_display, gcpp_true, use_rgb=True))
            axes[row, 4].set_title(f"Grad-CAM++ (true: {class_names[true_label]})", fontsize=10)
            axes[row, 4].axis('off')

    plt.tight_layout()
    plt.show()

# Misclassified: show both predicted and true class activations
wrong_indices = torch.where(~correct_mask)[0].tolist()
plot_gradcam(wrong_indices, "Grad-CAM — Misclassified Samples (predicted vs true class)", show_true_class=True)

## 4. Grad-CAM: Correctly Classified Samples

One random correct sample per class to see what the model focuses on when it gets it right.

In [ ]:
rng = np.random.default_rng(42)
correct_indices = []
for cls_idx in range(len(class_names)):
    cls_correct = torch.where(correct_mask & (all_labels == cls_idx))[0].tolist()
    if cls_correct:
        correct_indices.append(rng.choice(cls_correct))

plot_gradcam(correct_indices, "Grad-CAM — Correctly Classified Samples (1 per class)", show_true_class=False)